# MLflow experiment demo

This notebook **creates an MLflow experiment**, logs a run with metrics the LibreQuant **Experiments** page understands, and saves an **equity curve** artifact for the in-app preview.

**Requirements**

- **Docker Compose** with the `mlflow` service running, and Jupyter using `MLFLOW_TRACKING_URI` (set in `docker-compose.yml` for the `jupyter` service: `http://mlflow:5000`).
- **`librequant[mlflow]`** installed in the kernel (Compose installs `postgres` + `mlflow` extras on container start).

After you run the cells below, open **Experiments** in the app (or use the sidebar **MLflow experiments** list). You should see an experiment named like **`notebook_demo_buyhold`** with your run. Section **2b** registers a toy sklearn model to the **MLflow Model Registry** (visible under **Models** in the MLflow UI).

**Troubleshooting:** **403** in the browser or from `mlflow` usually means the tracking server’s **Host** allowlist is wrong. This repo merges MLflow’s **default** patterns with `mlflow` / `mlflow:5000` — do not replace that with a short `--allowed-hosts` list only. Restart the **`mlflow`** container after `docker-compose.yml` changes. **`PermissionError`** under `/mlflow` when logging artifacts: delete the experiment in the MLflow UI and re-run with an updated `librequant` (proxied **`mlflow-artifacts:/`**).


## 1. Environment check

Inside Compose, the tracking URI should point at the `mlflow` service. On the host (e.g. running Jupyter outside Docker), use `http://127.0.0.1:5000` if MLflow is exposed there.


In [ ]:
import os

uri = os.environ.get("MLFLOW_TRACKING_URI", "").strip()
print("MLFLOW_TRACKING_URI:", uri or "(not set — logging will fail until this is set)")

## 2. Run a tracked experiment

- **`strategy=`** becomes the **MLflow experiment name** (and shows in the sidebar experiment list).
- Params **`symbol`**, **`start_date`**, **`end_date`** are set automatically from the `backtest_run(...)` arguments.
- Metrics **`sharpe`**, **`max_drawdown`**, **`cagr`**, **`win_rate`** match what the Next.js UI maps into the runs table.
- **`set_equity_series`** writes `curves/equity_curve.csv` for the equity preview in the explorer.


In [ ]:
import numpy as np
import pandas as pd

from librequant.data import get_bars
from librequant.tracking import backtest_run

# Experiment name in MLflow (unique label in the sidebar + Experiments page)
STRATEGY = "notebook_demo_buyhold"

SYMBOL = "SPY"
START = "2023-01-01"
END = "2024-01-01"


def buy_and_hold_metrics(close: pd.Series) -> tuple[dict[str, float], pd.Series]:
    """Toy metrics from daily close: buy at first bar, hold."""
    close = close.astype(float).dropna()
    if len(close) < 5:
        raise ValueError("Not enough price data")

    daily_ret = close.pct_change().dropna()
    equity = close / float(close.iloc[0])  # normalized to 1.0

    if daily_ret.std() > 0:
        sharpe = float(np.sqrt(252) * daily_ret.mean() / daily_ret.std())
    else:
        sharpe = 0.0

    roll_max = equity.cummax()
    drawdown = equity / roll_max - 1.0
    max_drawdown = float(-drawdown.min())  # positive fraction, e.g. 0.18 = 18%

    days = (close.index[-1] - close.index[0]).days
    years = days / 365.25 if days > 0 else 0.0
    if years > 0:
        cagr = float(
            (float(close.iloc[-1]) / float(close.iloc[0])) ** (1.0 / years) - 1.0
        )
    else:
        cagr = 0.0

    win_rate = float((daily_ret > 0).mean())

    metrics = {
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "cagr": cagr,
        "win_rate": win_rate,
    }
    equity_series = equity.copy()
    equity_series.name = "equity"
    return metrics, equity_series


with backtest_run(
    strategy=STRATEGY,
    symbol=SYMBOL,
    start=START,
    end=END,
    params={"model": "buy_and_hold", "interval": "1d", "data_source": "yfinance"},
    tags={"notebook": "MlflowExperimentDemo.ipynb"},
) as run:
    bars = get_bars(SYMBOL, START, END, source="yfinance", interval="1d")
    metrics, equity = buy_and_hold_metrics(bars["close"])
    run.log_metrics(metrics)
    run.set_equity_series(equity)

print("Finished MLflow run for experiment:", STRATEGY)
print("Metrics:", metrics)

## 2b. Register a model (Model Registry)

After logging a model with a flavor such as `mlflow.sklearn.log_model`, register it so **Models** in the MLflow UI (and the API) can serve versions. Call **registration while the run is still active** (before the `with` block ends), or pass `run_id=` from a completed run.

Use `ensure_tracked_experiment(...)` when you start runs with plain `mlflow.start_run()` so the experiment uses proxied artifact uploads (same as `backtest_run`).

Requires **`librequant[mlflow]`** with scikit-learn for this sklearn example. Pass **`input_example`** to `log_model` so MLflow infers the model signature (avoids the “logged without a signature” warning).

In [ ]:
import numpy as np
import mlflow
from sklearn.linear_model import LinearRegression

from librequant.tracking import ensure_tracked_experiment
from librequant.mlflow_registry import register_model_from_run

REG_MODEL_EXPERIMENT = "notebook_demo_sklearn_registry"
REG_MODEL_NAME = "notebook_demo_linear_regression"  # name in MLflow Models

ensure_tracked_experiment(REG_MODEL_EXPERIMENT)

rng = np.random.default_rng(0)
X = np.arange(20, dtype=float).reshape(-1, 1)
y = 0.5 * X.ravel() + 3.0 + rng.normal(0.0, 0.1, size=20)
sk_model = LinearRegression().fit(X, y)

with mlflow.start_run(run_name="train_and_register") as run:
    mlflow.log_param("fit", "linear_regression")
    mlflow.sklearn.log_model(sk_model, "model", input_example=X[:5])
    mv = register_model_from_run(REG_MODEL_NAME, artifact_path="model")

print("Registered model:", REG_MODEL_NAME, "version:", mv.version)

## 3. Where to look in the app

1. **Sidebar → MLflow experiments** — your experiment name should appear; click it to open **Explorer** with that experiment selected.
2. **Experiments** page — pick the experiment in the dropdown, inspect runs, equity preview, and (optional) **Open MLflow UI** for the full MLflow server.
3. **MLflow UI → Models** — after section 2b, you should see the registered model name (e.g. `notebook_demo_linear_regression`) with a new version linked to the run.
